In [ ]:
import pandas as pd
import sys
import importlib

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
sys.path.append("../src")
import modeling as mod
import visualization as visual
import grid_search as gs

importlib.reload(mod)
importlib.reload(visual)
importlib.reload(gs)

In [ ]:
X_train_pp = pd.read_csv("../data/processed/one_hot/X_train_encoded.csv").drop(columns = ["Descripción", "Versión", "Título"])
X_val_pp = pd.read_csv("../data/processed/one_hot/X_val_encoded.csv").drop(columns = ["Descripción", "Versión", "Título"])

# Datasests pre one-hot for cross validation
X_train = pd.read_csv("../data/processed/splitted/X_train.csv").drop(columns = ["Descripción", "Versión", "Título"])
X_val = pd.read_csv("../data/processed/splitted/X_val.csv").drop(columns = ["Descripción", "Versión", "Título"])

y_train = pd.read_csv("../data/processed/splitted/y_train.csv").squeeze()
y_val = pd.read_csv("../data/processed/splitted/y_val.csv").squeeze()

<h1 style="
    background-color: #d0ebff; 
    color: #1a1a1a; 
    display: inline-block; 
    padding: 10px 18px; 
    border-radius: 10px;
    font-size: 32px;
">
Predictive Modeling
</h1>

En esta sección se entrenan y evalúan distintos modelos de regresión para estimar el precio de SUVs en dólares. Para cada familia de modelos se comparan dos estrategias de entrenamiento: utilizar el precio original como variable objetivo y utilizar una transformación logarítmica del precio.

Como se observó durante el EDA, la variable `Precio` presenta una distribución sesgada hacia la derecha: la mayoría de los vehículos se concentra en rangos de precios medios, mientras que algunos autos considerablemente más caros generan una cola larga de valores extremos.

Por este motivo, además de entrenar los modelos sobre el precio original, también se evalúa el entrenamiento sobre `log(Precio)`. Esta transformación reduce la influencia de los valores extremos y puede facilitar el aprendizaje de relaciones proporcionales entre las características del vehículo y su precio.

Para mantener una comparación justa entre todos los modelos, las predicciones obtenidas en escala logarítmica se transforman nuevamente a dólares antes de calcular las métricas. De esta forma, todos los modelos se evalúan utilizando las mismas medidas de desempeño: MSE, RMSE, MAE y R² en la escala original del precio.

El objetivo final es analizar no solo qué modelo produce el menor error predictivo, sino también si la transformación del target mejora la capacidad de generalización y la estabilidad de las predicciones frente a valores extremos.

<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
Baseline Model: Linear Regression
</h3>

La **regresión lineal** se utiliza como modelo base (*baseline*) porque proporciona una primera referencia del desempeño predictivo del problema. Este modelo asume una relación lineal entre las variables de entrada y el precio del vehículo, por lo que sirve como punto de comparación para evaluar si modelos más complejos logran mejorar la predicción.

A continuación, la regresión lineal se entrena utilizando las dos estrategias de target definidas previamente: el **precio original en dólares** y **log(Precio)**. En ambos casos, las métricas se calculan en la escala original del precio para permitir una comparación directa.

<div style="
    text-align: center;
    background-color: rgba(0, 0, 0, 0.3);
    color: white;
    padding: 10px;
    border-radius: 8px;
    font-weight: bold;
">
Original Target
</div>

In [ ]:
linear_model, linear_metrics, linear_predictions = mod.train_linear_regression(
    X_train_pp, y_train, X_val_pp, y_val
)
linear_metrics

<div style="
    text-align: center;
    background-color: rgba(0, 0, 0, 0.3);
    color: white;
    padding: 10px;
    border-radius: 8px;
    font-weight: bold;
">
Log Target
</div>

In [ ]:
linear_log_model, linear_log_metrics, linear_log_predictions = mod.train_linear_regression(
    X_train_pp, y_train, X_val_pp, y_val, use_log_target=True
)
linear_log_metrics

<div style="
    text-align: center;
    background-color: rgba(0, 0, 0, 0.3);
    color: white;
    padding: 10px;
    border-radius: 8px;
    font-weight: bold;
">
Comparison
</div>

In [ ]:
visual.plot_regression_metrics_comparison(
    {
        "Original Target": linear_metrics,
        "Log Target": linear_log_metrics,
    },
    title="Linear Regression - Original vs Log Target"
)

### Linear Regression: análisis del resultado

Al comparar ambos enfoques de entrenamiento, se observa que el modelo entrenado sobre `log(Precio)` presenta un desempeño consistentemente mejor tanto en entrenamiento como en validación. Las métricas de error (MSE, RMSE y MAE) son menores, mientras que el coeficiente de determinación (R²) es superior en ambos conjuntos.

Además, la brecha entre las métricas de entrenamiento y validación se mantiene relativamente estable en las dos estrategias, lo que sugiere que no existe un problema severo de sobreajuste. Sin embargo, el entrenamiento sobre el target logarítmico logra una capacidad de generalización superior, ya que obtiene mejores resultados de validación de manera consistente.

Este comportamiento indica que la transformación `log(Precio)` resulta beneficiosa para este problema. Al reducir la influencia de los vehículos considerablemente más caros y disminuir la asimetría de la distribución del target, el modelo lineal consigue capturar de manera más efectiva las relaciones entre las características de los vehículos y su precio.

<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
Ridge Regression
</h3>

La **regresión Ridge** es una extensión de la regresión lineal que incorpora regularización L2. Esta término de penalización reduce la magnitud de los coeficientes del modelo, ayudando a controlar posibles problemas de sobreajuste o multicolinealidad entre variables.

En este caso, Ridge resulta relevante porque luego del One-Hot Encoding el dataset contiene una mayor cantidad de variables predictoras, especialmente asociadas a categorías como marca, modelo y color. Al igual que en la regresión lineal, se evalúan dos estrategias de entrenamiento: precio original y `log(Precio)`.

<div style="
    text-align: center;
    background-color: rgba(0, 0, 0, 0.3);
    color: white;
    padding: 10px;
    border-radius: 8px;
    font-weight: bold;
">
Original Target
</div>

In [ ]:
# Search for the best Ridge alpha using 5-fold cross-validation on the training set
ALPHAS_RIDGE = (0.005, 0.01, 0.1, 1, 10, 100, 500)

best_ridge_alpha, ridge_cv_results = gs.find_best_alpha_ridge(
    X_train_pp, y_train, alphas=ALPHAS_RIDGE, cv=5, use_log_target=False
)

print(f"Best Ridge alpha: {best_ridge_alpha}")
ridge_cv_results.style.hide(axis="index")

In [ ]:
ridge_model, ridge_metrics, ridge_predictions = mod.train_ridge_regression(
    X_train_pp, y_train, X_val_pp, y_val, alpha=best_ridge_alpha
)

ridge_metrics

<div style="
    text-align: center;
    background-color: rgba(0, 0, 0, 0.3);
    color: white;
    padding: 10px;
    border-radius: 8px;
    font-weight: bold;
">
Log Target
</div>

In [ ]:
# Search for the best Ridge alpha using 5-fold cross-validation on the log-transformed target
best_ridge_alpha, ridge_cv_results = gs.find_best_alpha_ridge(
    X_train_pp, y_train, alphas=ALPHAS_RIDGE, cv=5, use_log_target=True
)

print(f"Best Ridge alpha: {best_ridge_alpha}")
ridge_cv_results.style.hide(axis="index")

In [ ]:
ridge_log_model, ridge_log_metrics, ridge_log_predictions = mod.train_ridge_regression(
    X_train_pp, y_train, X_val_pp, y_val, alpha=best_ridge_alpha,  use_log_target=True
)

<div style="
    text-align: center;
    background-color: rgba(0, 0, 0, 0.3);
    color: white;
    padding: 10px;
    border-radius: 8px;
    font-weight: bold;
">
Comparison
</div>

In [ ]:
visual.plot_regression_metrics_comparison(
    {
        "Original Target": ridge_metrics,
        "Log Target": ridge_log_metrics,
    },
    title="Ridge Regression - Original vs Log Target"
)

### Ridge Regression: análisis del resultado

Ridge Regression se incorporó como una extensión de la regresión lineal con regularización L2. El objetivo de este modelo es penalizar coeficientes demasiado grandes para reducir el riesgo de overfitting, especialmente cuando existen muchas variables predictoras o alta multicolinealidad entre ellas.

Sin embargo, en este caso Ridge no produjo una mejora significativa respecto de la regresión lineal. Esto puede explicarse porque, aunque el dataset tiene muchas variables luego del One-Hot Encoding, la cantidad de observaciones es suficientemente grande en relación con la cantidad de features. Por lo tanto, el modelo lineal ya se encuentra relativamente bien determinado y no parece estar sufriendo un sobreajuste severo.

Al evaluar distintos valores de `alpha`, las métricas obtenidas fueron muy similares entre sí. Esto indica que aumentar la intensidad de la regularización no mejora de forma relevante la capacidad predictiva del modelo. En otras palabras, penalizar más los coeficientes no reduce sustancialmente el error de validación.

Por este motivo, se seleccionó el valor de `alpha` que obtuvo el menor MSE, aunque la diferencia con otros valores fue pequeña. Este resultado sugiere que, para este problema, Ridge Regression funciona principalmente como una confirmación de que la regresión lineal simple ya constituye una baseline estable.

<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
LASSO Regression
</h3>

<div style="
    text-align: center;
    background-color: rgba(0, 0, 0, 0.3);
    color: white;
    padding: 10px;
    border-radius: 8px;
    font-weight: bold;
">
Original Target
</div>

In [ ]:
# Search for the best LASSO alpha using 5-fold cross-validation on the original target
ALPHAS_LASSO = (0.1, 1, 10, 100, 500)

best_lasso_alpha, lasso_cv_results = gs.find_best_alpha_lasso(
    X_train_pp,
    y_train,
    alphas=ALPHAS_LASSO,
    cv=5,
    use_log_target=False,
)

In [ ]:
print(f"Best LASSO alpha: {best_lasso_alpha}")
lasso_cv_results.style.hide(axis="index")

In [ ]:
lasso_model, lasso_metrics, lasso_predictions = mod.train_lasso_regression(
    X_train_pp, y_train, X_val_pp, y_val, alpha=best_lasso_alpha, use_log_target=False
)
lasso_metrics

<div style="
    text-align: center;
    background-color: rgba(0, 0, 0, 0.3);
    color: white;
    padding: 10px;
    border-radius: 8px;
    font-weight: bold;
">
Log Target
</div>

In [ ]:
# Search for the best LASSO alpha using 5-fold cross-validation on the log-transformed target
best_lasso_log_alpha, lasso_log_cv_results = gs.find_best_alpha_lasso(
    X_train_pp,
    y_train,
    alphas=ALPHAS_LASSO,
    cv=5,
    use_log_target=True,
)

print(f"Best LASSO alpha: {best_lasso_log_alpha}")
lasso_log_cv_results.style.hide(axis="index")

In [ ]:
lasso_log_model, lasso_log_metrics, lasso_log_predictions = mod.train_lasso_regression(
    X_train_pp, y_train, X_val_pp, y_val, alpha=best_lasso_alpha, use_log_target=True
)
lasso_metrics

<div style="
    text-align: center;
    background-color: rgba(0, 0, 0, 0.3);
    color: white;
    padding: 10px;
    border-radius: 8px;
    font-weight: bold;
">
Comparison
</div>

In [ ]:
visual.plot_regression_metrics_comparison(
    {
        "Original Target": lasso_metrics,
        "Log Target": lasso_log_metrics,
    },
    title="LASSO Regression - Original vs Log Target"
)

<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
Random Forest Regressor
</h3>

El **Random Forest Regressor** es un modelo de ensamble basado en múltiples árboles de decisión. Cada árbol se entrena sobre una muestra distinta del conjunto de datos y, en regresión, la predicción final se obtiene promediando las predicciones individuales.

Este modelo permite capturar relaciones no lineales e interacciones entre variables sin tener que definirlas manualmente. Además, suele ser más robusto que un único árbol de decisión porque reduce la varianza mediante el promedio de varios árboles.

A continuación, se ajustan sus hiperparámetros usando una búsqueda sobre distintas combinaciones. Para evitar correr este proceso costoso en cada ejecución del notebook, los resultados de la búsqueda se guardan en archivos `.csv` y luego pueden reutilizarse.

<div style="
    text-align: center;
    background-color: rgba(0, 0, 0, 0.3);
    color: white;
    padding: 10px;
    border-radius: 8px;
    font-weight: bold;
">
Original Target
</div>

In [ ]:
RF_PARAM_GRID = {
    "max_depth": [24, 28, None],
    "max_features": [0.2, 0.3, 0.4],
    "min_samples_leaf": [1, 2],
    "min_samples_split": [2, 3],
    "n_estimators": [300, 500],
}

RUN_RF_SEARCH = False

In [ ]:
best_rf_params, rf_search_results = gs.run_or_load_random_forest_search(
    X_train=X_train_pp,
    y_train=y_train,
    param_grid=RF_PARAM_GRID,
    output_path="../results/random_forest/rf_original_target_search.csv",
    run_search=RUN_RF_SEARCH,
    cv=3,
    use_log_target=False,
    scoring="rmse",
)

best_rf_params

In [ ]:
rf_model, rf_metrics, rf_predictions = mod.train_random_forest(
    X_train_pp,
    y_train,
    X_val_pp,
    y_val,
    use_log_target=False,
    **best_rf_params,
)

rf_metrics

<div style="
    text-align: center;
    background-color: rgba(0, 0, 0, 0.3);
    color: white;
    padding: 10px;
    border-radius: 8px;
    font-weight: bold;
">
Log Target
</div>

In [ ]:
RF_LOG_PARAM_GRID = {
    "max_depth": [24, 26, 28],
    "max_features": [0.45, 0.5, 0.55],
    "min_samples_leaf": [1],
    "min_samples_split": [2, 3],
    "n_estimators": [300, 500]
}

RUN_RF_LOG_SEARCH = False

In [ ]:
best_rf_log_params, rf_log_search_results = gs.run_or_load_random_forest_search(
    X_train=X_train_pp,
    y_train=y_train,
    param_grid=RF_LOG_PARAM_GRID,
    output_path="../results/random_forest/rf_log_target_search.csv",
    run_search=RUN_RF_LOG_SEARCH,
    cv=3,
    use_log_target=True,
    scoring="rmse",
)

best_rf_log_params

In [ ]:
rf_log_model, rf_log_metrics, rf_log_predictions = mod.train_random_forest(
    X_train_pp,
    y_train,
    X_val_pp,
    y_val,
    use_log_target=True,
    **best_rf_log_params,
)

rf_log_metrics

<div style="
    text-align: center;
    background-color: rgba(0, 0, 0, 0.3);
    color: white;
    padding: 10px;
    border-radius: 8px;
    font-weight: bold;
">
Comparison
</div>

In [ ]:
visual.plot_regression_metrics_comparison(
    {
        "Original Target": rf_metrics,
        "Log Target": rf_log_metrics,
    },
    title="Random Forest - Original vs Log Target"
)

### Random Forest: análisis del resultado

Random Forest se incorporó porque es un modelo basado en ensambles de árboles de decisión capaz de capturar relaciones no lineales e interacciones complejas entre las variables sin necesidad de especificarlas manualmente. Además, suele ser robusto frente a outliers y se adapta particularmente bien a problemas de regresión sobre datos tabulares.

Los resultados muestran una mejora muy importante respecto de los modelos lineales. Tanto el error de entrenamiento como el de validación son considerablemente menores, mientras que el coeficiente de determinación (R²) es notablemente más alto. Esto sugiere que las relaciones entre las características de los vehículos y su precio no son puramente lineales y que el modelo se beneficia de poder aprender patrones más complejos.

Al comparar el entrenamiento sobre el precio original y sobre `log(Precio)`, se observa que ambos enfoques producen desempeños muy similares. Sin embargo, el modelo entrenado sobre el precio original obtiene sistemáticamente errores de validación ligeramente menores y un R² apenas superior.

Además, la diferencia entre las métricas de entrenamiento y validación se mantiene relativamente acotada en ambos casos, lo que indica que el modelo logra una buena capacidad de generalización y no presenta señales de overfitting.

Por este motivo, se selecciona el modelo entrenado sobre el precio original como la mejor configuración de Random Forest para este problema. Los resultados obtenidos sugieren que el ensamble de árboles logra modelar adecuadamente la complejidad del dataset y constituye uno de los principales candidatos a obtener el mejor desempeño predictivo final.

<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
XGBoost
</h3>

<div style="
    text-align: center;
    background-color: rgba(0, 0, 0, 0.3);
    color: white;
    padding: 10px;
    border-radius: 8px;
    font-weight: bold;
">
Original Target
</div>

In [ ]:
XGB_PARAM_GRID = {
    "n_estimators": [300, 500],
    "learning_rate": [0.03, 0.05],
    "max_depth": [4, 6],
    "min_child_weight": [1, 3],
    "subsample": [0.8],
    "colsample_bytree": [0.8],
    "reg_alpha": [0.0],
    "reg_lambda": [1.0],
}

RUN_XGB_SEARCH = True

In [ ]:
best_xgb_params, xgb_search_results = gs.run_or_load_xgboost_search(
    X_train=X_train_pp,
    y_train=y_train,
    param_grid=XGB_PARAM_GRID,
    output_path="../results/xgboost/xgb_original_target_search.csv",
    run_search=RUN_XGB_SEARCH,
    cv=3,
    use_log_target=False,
    scoring="rmse",
)

best_xgb_params

In [ ]:
xgb_model, xgb_metrics, xgb_predictions = mod.train_xgboost(
    X_train_pp, y_train, X_val_pp, y_val, use_log_target=False, **best_xgb_params
)
xgb_metrics

<div style="
    text-align: center;
    background-color: rgba(0, 0, 0, 0.3);
    color: white;
    padding: 10px;
    border-radius: 8px;
    font-weight: bold;
">
Log Target
</div>

In [ ]:
RUN_XGB_LOG_SEARCH = True

In [ ]:
best_xgb_log_params, xgb_log_search_results = gs.run_or_load_xgboost_search(
    X_train=X_train_pp,
    y_train=y_train,
    param_grid=XGB_PARAM_GRID,
    output_path="../results/xgboost_log_target_search.csv",
    run_search=RUN_XGB_LOG_SEARCH,
    cv=3,
    use_log_target=True,
    scoring="rmse",
)

best_xgb_log_params

In [ ]:
xgb_log_model, xgb_log_metrics, xgb_log_predictions = mod.train_xgboost(
    X_train_pp,
    y_train,
    X_val_pp,
    y_val,
    use_log_target=True,
    **best_xgb_log_params,
)

xgb_log_metrics

<div style="
    text-align: center;
    background-color: rgba(0, 0, 0, 0.3);
    color: white;
    padding: 10px;
    border-radius: 8px;
    font-weight: bold;
">
Comparison
</div>

In [ ]:
visual.plot_regression_metrics_comparison(
    {
        "Original Target": xgb_metrics,
        "Log Target": xgb_log_metrics,
    },
    title="XG Boost - Original vs Log Target"
)

In [ ]:
visual.plot_regression_metrics(
    xgboost_metrics,
    model_name="XGBoost Metrics",
)

In [ ]:
visual.plot_regression_metrics(
    xgboost_log_metrics,
    model_name="XGBoost Log",
)

<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
Neuronal Netowrk
</h3>

In [ ]:
nn_model, nn_metrics, nn_predictions = mod.train_neural_network(
    X_train_pp,
    y_train,
    X_val_pp,
    y_val,
    use_log_target=True,
    hidden_layer_sizes=(128, 64),
    activation="relu",
    alpha=0.001,
    learning_rate_init=0.001,
    batch_size=64,
    max_iter=500,
)

nn_metrics

<h1 style="
    background-color: #d0ebff; 
    color: #1a1a1a; 
    display: inline-block; 
    padding: 10px 18px; 
    border-radius: 10px;
    font-size: 32px;
">
Errors Analysis
</h1>

In [ ]:
xgboost_log_error_context = mod.attach_prediction_context(
    xgboost_log_predictions,
    X_train,
    X_val,
)

mod.top_prediction_errors(
    xgboost_log_error_context,
    split="validation",
    n=20,
).style.hide(axis="index")

In [ ]:
mod.summarize_prediction_errors(
    xgboost_log_error_context,
    group_cols="Marca",
    split="validation",
    min_count=20,
)

SEPARO EN ALTA GAMA Y PRUEBO

In [ ]:
high_end_brands = [
    "Marca_alfa romeo",
    "Marca_audi",
    "Marca_bmw",
    "Marca_ds",
    "Marca_jaguar",
    "Marca_land rover",
    "Marca_lexus",
    "Marca_mercedes benz",
    "Marca_mini",
    "Marca_porsche",
    "Marca_volvo",
]

In [ ]:
high_end_mask_train = X_train_pp[high_end_brands].any(axis=1)
high_end_mask_val = X_val_pp[high_end_brands].any(axis=1)

In [ ]:
X_train_high_end = X_train_pp[high_end_mask_train]
X_train_rest = X_train_pp[~high_end_mask_train]
y_train_high_end = y_train[high_end_mask_train]
y_train_rest = y_train[~high_end_mask_train]

In [ ]:
X_val_high_end = X_val_pp[high_end_mask_val]
X_val_rest = X_val_pp[~high_end_mask_val]
y_val_high_end = y_val[high_end_mask_val]
y_val_rest = y_val[~high_end_mask_val]

In [ ]:
xgboost_log_model, xgboost_log_metrics, xgboost_log_predictions = mod.train_xgboost(
    X_train_rest,
    y_train_rest,
    X_val_rest,
    y_val_rest,
    use_log_target=True,
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
)

In [ ]:
visual.plot_regression_metrics(
    xgboost_log_metrics,
    model_name="XGBoost Log",
)

In [ ]:
xgboost_log_model, xgboost_log_metrics, xgboost_log_predictions = mod.train_xgboost(
    X_train_high_end,
    y_train_high_end,
    X_val_high_end,
    y_val_high_end,
    use_log_target=True,
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
)

In [ ]:
visual.plot_regression_metrics(
    xgboost_log_metrics,
    model_name="XGBoost Log",
)